# Phase 4 — Opportunity Pipeline: Interactive Test

This notebook tests the **Phase 4 Opportunity Pipeline** end-to-end.

## How it works
1. You define an opportunity (name + prompt)
2. The notebook can either:
   - **Direct call** the test-runner server (`/run-opportunity`) — fast, no n8n involved
   - **Drop a file** into `opportunities/pending/` — triggers the n8n file watcher
3. Output files land in `~/output/projects/<name>/`

## Architecture recap
```
opportunities/pending/<name>.json
         │
         ▼  (n8n localFileTrigger)
opportunity_intake.js  →  pending/ → running/
         │
         ▼  (HTTP POST)
test-runner:5001/run-opportunity
         │  1. Plan  (hybrid/chat)
         │  2. Code  (free/code)
         │  3. Validate & fix format
         │  4. Extract files  →  output/projects/<name>/project/
         │  5. Postprocess
         │  6. Test & fix loop (free/code, up to 3 attempts)
         │  7. Write phase4 report
         ▼
opportunities/done/<name>.json
```

In [1]:
import json, os, re, time
from pathlib import Path
import requests

# ── config ────────────────────────────────────────────────────────────────────
TEST_RUNNER_URL = 'http://test-runner:5001'          # internal Docker network
OUTPUT_ROOT     = Path('/home/jovyan/output')         # mounted from ./output
OPPORTUNITIES   = OUTPUT_ROOT / 'opportunities'       # shared with n8n
OUTPUT_BASE     = OUTPUT_ROOT / 'projects'

# ensure directories exist
for d in ['pending', 'running', 'done', 'failed']:
    (OPPORTUNITIES / d).mkdir(parents=True, exist_ok=True)

# health check
try:
    r = requests.get(f'{TEST_RUNNER_URL}/health', timeout=5)
    print('test-runner health:', r.json())
except Exception as e:
    print(f'ERROR: test-runner not reachable — {e}')
    print('Make sure the stack is running: docker compose up -d')


test-runner health: {'status': 'ok'}


## 1 · Define your opportunity

In [2]:
# ── edit these ────────────────────────────────────────────────────────────────
OPPORTUNITY_NAME   = 'calculator-lib'
OPPORTUNITY_PROMPT = (
    'Build a Python library with a Calculator class that supports '
    'add(a, b), subtract(a, b), multiply(a, b), and divide(a, b) methods. '
    'Include comprehensive pytest tests covering normal cases and edge cases '
    '(e.g. division by zero raises ValueError).'
)

print(f'Name  : {OPPORTUNITY_NAME}')
print(f'Prompt: {OPPORTUNITY_PROMPT[:100]}...')

Name  : calculator-lib
Prompt: Build a Python library with a Calculator class that supports add(a, b), subtract(a, b), multiply(a, ...


## 2 · Option A — Direct call (bypasses n8n)

Calls the test-runner directly — useful for debugging the pipeline without n8n.

In [3]:
safe_name    = re.sub(r'[^a-z0-9-_]', '-', OPPORTUNITY_NAME.lower())
safe_name    = re.sub(r'-+', '-', safe_name).strip('-')
project_base = OUTPUT_BASE / safe_name
project_dir  = project_base / 'project'
plan_path    = project_base / 'project_plan.md'
code_path    = project_base / 'execution_output.md'
report_path  = project_base / 'phase4_report.md'

# test-runner sees /data/output/... — same volume, different mount point
def to_runner_path(p):
    return str(p).replace('/home/jovyan/output', '/data/output')

payload = {
    'name':             safe_name,
    'prompt':           OPPORTUNITY_PROMPT,
    'project_base':     to_runner_path(project_base),
    'project_dir':      to_runner_path(project_dir),
    'plan_path':        to_runner_path(plan_path),
    'code_output_path': to_runner_path(code_path),
    'report_path':      to_runner_path(report_path),
}

print(f'Calling {TEST_RUNNER_URL}/run-opportunity ...')
print(f'Project will be at: {project_dir}')
print('This may take 2–5 minutes ...\n')

t0 = time.time()
try:
    resp = requests.post(
        f'{TEST_RUNNER_URL}/run-opportunity',
        json=payload,
        timeout=1200,
    )
    elapsed = time.time() - t0
    print(f'Status: {resp.status_code}  ({elapsed:.0f}s)')

    if resp.status_code == 200:
        result = resp.json()
        icon   = '✅ PASSED' if result.get('passed') else '❌ FAILED'
        print(f'Result: {icon}')
        print(f'Iterations: {result.get("iterations")}')
        print(f'Project dir: {result.get("project_dir")}')
        print('\n── Pipeline log ──')
        for line in result.get('log', []):
            print(line)
    else:
        print('ERROR response:')
        print(resp.text[:2000])
except requests.exceptions.Timeout:
    print(f'Timed out after {time.time()-t0:.0f}s — check: docker logs test_runner')
except Exception as e:
    print(f'ERROR: {e}')


Calling http://test-runner:5001/run-opportunity ...
Project will be at: /home/jovyan/output/projects/calculator-lib/project
This may take 2–5 minutes ...



Status: 200  (226s)
Result: ✅ PASSED
Iterations: 1
Project dir: /data/output/projects/calculator-lib/project

── Pipeline log ──
=== Phase 4 — calculator-lib ===
Step 1: Generating project plan ...
  Plan saved (2947 chars)
Step 2: Generating code ...
  Code output saved (3206 chars)
Step 3: Validating format ...
  valid=True files=6 issues=[]
Step 4: Extracting files ...
  Wrote 6 files: ['calculator_library/calculator/calculator.py', 'calculator_library/tests/test_calculator.py', 'calculator_library/requirements.txt', 'calculator_library/Dockerfile', 'calculator_library/.gitignore', 'calculator_library/README.md']
Step 5: Running postprocess ...
Step 6: Running test & fix loop ...
  venv: Deps unchanged — skipped install
  ✅ All tests passed on attempt 1!

Report → /data/output/projects/calculator-lib/phase4_report.md


## 3 · Option B — Drop a file into opportunities/pending/ (triggers n8n)

This exercises the full workflow including the file-watcher, `opportunity_intake.js`, and the n8n pipeline.

In [4]:
# Write an opportunity file to pending/
opp_file = OPPORTUNITIES / 'pending' / f'{OPPORTUNITY_NAME}.json'
opp_data = {'name': OPPORTUNITY_NAME, 'prompt': OPPORTUNITY_PROMPT}
opp_file.write_text(json.dumps(opp_data, indent=2))

print(f'Wrote: {opp_file}')
print('n8n file watcher should pick this up within a few seconds.')
print()
print('Monitor progress:')
print('  docker logs -f test_runner')
print('  docker logs -f n8n')
print()
print('Expected lifecycle: pending/ → running/ → done/')


Wrote: /home/jovyan/output/opportunities/pending/calculator-lib.json
n8n file watcher should pick this up within a few seconds.

Monitor progress:
  docker logs -f test_runner
  docker logs -f n8n

Expected lifecycle: pending/ → running/ → done/


In [5]:
# Poll for completion (runs up to 20 minutes)
fname = f'{OPPORTUNITY_NAME}.json'

print(f'Polling for {fname} ...')
for i in range(120):
    time.sleep(10)
    if (OPPORTUNITIES / 'done' / fname).exists():
        print(f'\n✅ Done! Moved to done/')
        break
    if (OPPORTUNITIES / 'failed' / fname).exists():
        print(f'\n❌ Failed! Moved to failed/')
        break
    status = 'running' if (OPPORTUNITIES / 'running' / fname).exists() else 'pending'
    print(f'  [{i*10}s] {status}', end='\r')
else:
    print('\nTimed out after 20 minutes')


Polling for calculator-lib.json ...



✅ Done! Moved to done/


## 4 · Inspect outputs

In [6]:
safe_name    = re.sub(r'[^a-z0-9-_]', '-', OPPORTUNITY_NAME.lower())
safe_name    = re.sub(r'-+', '-', safe_name).strip('-')
project_base = OUTPUT_BASE / safe_name

print(f'Project base: {project_base}\n')

if project_base.exists():
    SKIP_DIRS = {'.venv', '__pycache__', '.pytest_cache', '.git', 'node_modules'}
    for f in sorted(project_base.rglob('*')):
        if f.is_file() and not (SKIP_DIRS & set(f.relative_to(project_base).parts)):
            print(f'  {str(f.relative_to(project_base)):<55} {f.stat().st_size:>8,} bytes')
else:
    print('Project directory does not exist yet — run the pipeline first.')


Project base: /home/jovyan/output/projects/calculator-lib

  execution_output.md                                        3,206 bytes
  phase4_report.md                                           1,061 bytes
  project/.gitignore                                            41 bytes
  project/Dockerfile                                           151 bytes
  project/README.md                                            597 bytes
  project/calculator.py                                        308 bytes
  project/calculator_library/.gitignore                         48 bytes
  project/calculator_library/Dockerfile                        523 bytes
  project/calculator_library/README.md                         508 bytes
  project/calculator_library/__init__.py                        61 bytes
  project/calculator_library/calculator/calculator.py          763 bytes
  project/calculator_library/calculator.py                     308 bytes
  project/calculator_library/requirements.txt                    

In [7]:
# Display the phase4 report (uses path from Option A payload)
report = project_base / 'phase4_report.md'
# Force re-read to avoid cached content
import importlib
report_text = report.read_text() if report.exists() else ''
if not report_text:
    print('No phase4_report.md found yet.')
else:
    print(report_text)


# Phase 4 Report — ✅ PASSED

**Project**: `calculator-lib`  
**Result**: ✅ PASSED  
**Iterations**: 1

## Attempt 1 — ✅ PASS
```
============================= test session starts ==============================
collecting ... collected 9 items

tests/test_calculator.py::test_add PASSED                                [ 11%]
tests/test_calculator.py::test_subtract PASSED                           [ 22%]
tests/test_calculator.py::test_multiply PASSED                           [ 33%]
tests/test_calculator.py::test_divide PASSED                             [ 44%]
tests/test_calculator.py::test_divide_by_zero PASSED                     [ 55%]
tests/test_calculator.py::test_add_negative_numbers PASSED               [ 66%]
tests/test_calculator.py::test_subtract_negative_numbers PASSED          [ 77%]
tests/test_calculator.py::test_multiply_negative_numbers PASSED          [ 88%]
tests/test_calculator.py::test_divide_negative_numbers PASSED            [100%]

============================== 9 pa

In [8]:
# Read the project plan
plan = project_base / 'project_plan.md'
if plan.exists():
    print(plan.read_text()[:3000])
else:
    print('No plan found yet.')

**Project Overview**

Project Name: Calculator Library

Project Description: A Python library that provides a Calculator class with basic arithmetic operations (addition, subtraction, multiplication, and division).

**Technical Stack**

* Programming Language: Python 3.9
* Testing Framework: Pytest
* Type Hinting: Python 3.5+

**File Structure**

```bash
calculator_library/
calculator/
__init__.py
calculator.py
tests/
__init__.py
test_calculator.py
requirements.txt
README.md
```

**Core Components**

1. `calculator/calculator.py`: The Calculator class with `add`, `subtract`, `multiply`, and `divide` methods.
2. `tests/test_calculator.py`: Pytest tests for the Calculator class.

**Implementation Steps**

### Calculator Class

`calculator/calculator.py`:
```python
from typing import Union

class Calculator:
    def add(self, a: Union[int, float], b: Union[int, float]) -> Union[int, float]:
        """Return the sum of a and b."""
        return a + b

    def subtract(self, a: Union[int,

In [9]:
# List generated project files
proj_dir = project_base / 'project'
if proj_dir.exists():
    SKIP_DIRS = {'.venv', '__pycache__', '.pytest_cache', '.git', 'node_modules'}
    for f in sorted(proj_dir.rglob('*')):
        if f.is_file() and not (SKIP_DIRS & set(f.relative_to(proj_dir).parts)):
            print(str(f.relative_to(proj_dir)))
else:
    print('Project directory not created yet.')


.gitignore
Dockerfile
README.md
calculator.py
calculator_library/.gitignore
calculator_library/Dockerfile
calculator_library/README.md
calculator_library/__init__.py
calculator_library/calculator/calculator.py
calculator_library/calculator.py
calculator_library/requirements.txt
calculator_library/tests/test_calculator.py
requirements.txt
tests/conftest.py
tests/test_calculator.py


## 5 · Opportunity queue management

In [10]:
# View all opportunity states
print('=== Opportunity Queue ===')
for state in ('pending', 'running', 'done', 'failed'):
    d = OPPORTUNITIES / state
    files = sorted(d.glob('*.json')) if d.exists() else []
    print(f'\n{state.upper()}/ ({len(files)})')
    for f in files:
        try:
            data    = json.loads(f.read_text())
            started = data.get('startedAt', '')
            print(f'  {f.name:<35}  started={started}')
        except Exception:
            print(f'  {f.name}  (unreadable)')


=== Opportunity Queue ===

PENDING/ (0)

RUNNING/ (2)
  calculator-lib.json                  started=
  hello-world-api.json                 started=

DONE/ (3)
  calculator-lib.json                  started=2026-03-11T01:40:14.033Z
  hello-world-api.json                 started=2026-03-11T00:45:52.054Z
  nb-test.json                         started=2026-03-11T00:19:09.699Z

FAILED/ (0)


In [11]:
# ── Batch submit multiple opportunities ────────────────────────────────────────
# Uncomment and edit to queue multiple projects at once

# batch = [
#     {'name': 'todo-api',      'prompt': 'Build a REST API for todo list management with SQLite storage and pytest tests.'},
#     {'name': 'word-counter',  'prompt': 'Build a CLI tool that counts words, lines, and chars in text files with pytest tests.'},
#     {'name': 'csv-processor', 'prompt': 'Build a Python library that reads CSV files, filters rows, and exports results. Include tests.'},
# ]

# pending_dir = OPPORTUNITIES / 'pending'
# pending_dir.mkdir(parents=True, exist_ok=True)

# for opp in batch:
#     f = pending_dir / f"{opp['name']}.json"
#     f.write_text(json.dumps(opp, indent=2))
#     print(f'Queued: {f.name}')

print('Batch submit cell is ready — uncomment the code above to use it.')

Batch submit cell is ready — uncomment the code above to use it.


## 6 · Test-runner server utilities

In [12]:
# Docker logs: run this from your host terminal to stream logs:
#   docker logs -f test_runner
#   docker logs -f n8n
print('To stream live logs, run from host:')
print('  docker logs -f test_runner')
print('  docker logs -f n8n')


To stream live logs, run from host:
  docker logs -f test_runner
  docker logs -f n8n


In [13]:
# Manually check free/code model availability via LiteLLM
LITELLM_URL = 'http://litellm:4000'
LITELLM_KEY = os.environ.get('LITELLM_API_KEY', 'sk-sa-prod-ce5d031e2a50ffa45d3a200c037971f81853e27ed19b894bc3630625cba0b71a')

try:
    r = requests.get(
        f'{LITELLM_URL}/v1/models',
        headers={'Authorization': f'Bearer {LITELLM_KEY}'},
        timeout=10,
    )
    models = r.json().get('data', [])
    free_models = [m['id'] for m in models if m['id'].startswith('free/')]
    print(f'Free models available ({len(free_models)}):')
    for m in sorted(free_models):
        print(f'  {m}')
    if not free_models:
        print('No free models found — run free_model_sync.py to populate them')
        print('docker exec free_model_sync python /app/free_model_sync.py')
except Exception as e:
    print(f'Cannot reach LiteLLM: {e}')

Free models available (35):
  free/chat
  free/code
  free/dolphin-mistral-24b-venice-edition:free
  free/fast
  free/free
  free/gemini-2.0-flash
  free/gemini-2.0-flash-lite
  free/gemma-3-12b-it:free
  free/gemma-3-27b-it:free
  free/gemma-3-4b-it:free
  free/gemma-3n-e2b-it:free
  free/gemma-3n-e4b-it:free
  free/glm-4.5-air:free
  free/gpt-oss-120b:free
  free/gpt-oss-20b:free
  free/hermes-3-llama-3.1-405b:free
  free/lfm-2.5-1.2b-instruct:free
  free/lfm-2.5-1.2b-thinking:free
  free/llama-3.1-8b-instant
  free/llama-3.2-3b-instruct:free
  free/llama-3.3-70b-instruct:free
  free/llama-3.3-70b-versatile
  free/mistral-small-3.1-24b-instruct:free
  free/nemotron-3-nano-30b-a3b:free
  free/nemotron-nano-12b-v2-vl:free
  free/nemotron-nano-9b-v2:free
  free/qwen3-4b:free
  free/qwen3-coder:free
  free/qwen3-next-80b-a3b-instruct:free
  free/qwen3-vl-235b-a22b-thinking
  free/qwen3-vl-30b-a3b-thinking
  free/reason
  free/step-3.5-flash:free
  free/trinity-large-preview:free
  free/t

In [14]:
# Quick smoke-test: call free/code with a tiny prompt to verify it works
LITELLM_URL = 'http://litellm:4000'
LITELLM_KEY = os.environ.get('LITELLM_API_KEY', 'sk-sa-prod-ce5d031e2a50ffa45d3a200c037971f81853e27ed19b894bc3630625cba0b71a')

print('Testing free/code model ...')
t0 = time.time()
try:
    r = requests.post(
        f'{LITELLM_URL}/v1/chat/completions',
        headers={'Authorization': f'Bearer {LITELLM_KEY}', 'Content-Type': 'application/json'},
        json={
            'model':      'hybrid/fast',   # hybrid: tries local first, falls back to cloud
            'max_tokens': 64,
            'messages':   [{'role': 'user', 'content': 'Write a Python one-liner that prints hello world.'}],
        },
        timeout=30,
    )
    elapsed = time.time() - t0
    if r.status_code == 200:
        content = r.json()['choices'][0]['message']['content']
        print(f'OK ({elapsed:.1f}s): {content.strip()[:200]}')
    else:
        print(f'HTTP {r.status_code}: {r.text[:500]}')
except Exception as e:
    print(f'ERROR: {e}')

Testing free/code model ...


OK (0.1s): `print("hello world")`
